#### This notebook will form a tool having following capibilities:
1) Add Docstring 
2) Form test case

Frontier model: 
1) gpt-5-nano through openai
2) gemini-2.5-pro through gemini

open source models:
1) openai/gpt-oss-120b through groq
2) llama-3.3-70b-versatile through groq
3) gemma4:31b-cloud through ollama
4) qwen3.5:cloud through ollama


In [ ]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display
import json
import traceback


In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

In [ ]:
# Connect to client libraries

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"
groq_url = "https://api.groq.com/openai/v1"


# ---- CLIENT SETUP ----
openai_client = OpenAI()
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

models = [
    "gpt-5-nano",
    "gemini-2.5-flash", # Change this
    "openai/gpt-oss-120b",
    "llama-3.3-70b-versatile",
    "gemma4:31b-cloud",
    "qwen3-coder-next:cloud" # change this
]

clients = {
    "gpt-5-nano": openai_client,
    "gemini-2.5-flash": gemini,
    "openai/gpt-oss-120b": groq,
    "llama-3.3-70b-versatile": groq,
    "gemma4:31b-cloud": ollama,
    "qwen3-coder-next:cloud": ollama
}

In [ ]:
DOCS_SYSTEM_PROMPT = """
You are a senior Python developer.

Your task:
- Add clear docstrings to all functions and classes
- Add meaningful inline comments
- Do NOT change the logic of the code
- Do NOT rename variables or functions
- Keep code executable

Return ONLY the updated Python code.
"""


TESTCASE_SYSTEM_PROMPT = """
You are a strict Python testing expert.

Given a Python function:
1. Generate EXACTLY 5 test cases
2. Focus on edge cases and tricky inputs
3. Each list must contain at most 10 elements
4. DO NOT generate very large lists
5. Include:
   - Normal case
   - Edge case
   - Corner case
   - Invalid input (if applicable)
   - Stress/difficult case
6. Use ONLY standard ASCII characters
7. Do NOT use fancy quotes or special unicode
8. Do NOT wrap in markdown (no ```)

Return ONLY valid JSON. No markdown. No explanation.

Format:
[
  {
    "input": [args],
    "expected_output": value,
    "description": "..."
  }
]

IMPORTANT:
- Ensure expected_output is CORRECT
- No explanations outside JSON
"""

In [ ]:
def add_docs_and_comments(code, model_name):
    client = clients[model_name]

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": DOCS_SYSTEM_PROMPT},
            {"role": "user", "content": code}
        ]
    )

    return response.choices[0].message.content

In [ ]:
import re
import textwrap

def clean_code(code: str) -> str:
    # Remove markdown code fences
    code = re.sub(r"```python|```", "", code)

    # Normalize indentation
    code = textwrap.dedent(code)

    return code.strip()

In [ ]:
def generate_test_cases(code, model_name):
    client = clients[model_name]

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": TESTCASE_SYSTEM_PROMPT},
            {"role": "user", "content": code}
        ]
    )

    content = response.choices[0].message.content

    try:
        return json.loads(content)
    except:
        raise ValueError("Invalid JSON from model:\n" + content)

In [ ]:
def run_tests(code, test_cases, function_name):
    local_env = {}
    output_log = ""

    # ✅ CLEAN HERE
    code = clean_code(code)

    try:
        exec(code, local_env)
    except Exception:
        return "❌ Code execution failed:\n" + traceback.format_exc()

    func = local_env.get(function_name)

    if not func:
        return f"❌ Function '{function_name}' not found."

    for i, test in enumerate(test_cases, 1):
        inputs = test["input"]
        expected = test["expected_output"]
        desc = test.get("description", "")

        try:
            result = func(*inputs)

            # ✅ Handle expected exceptions
            if isinstance(expected, str) and "Error" in expected:
                passed = False
                output_log += f"\nTest Case {i}: {desc}\n"
                output_log += f"Input: {inputs}\n"
                output_log += f"Expected Exception: {expected}\n"
                output_log += f"Returned: {result}\n"
                output_log += "❌ FAIL (Expected exception but got value)\n"
                continue

            passed = result == expected

            output_log += f"\nTest Case {i}: {desc}\n"
            output_log += f"Input: {inputs}\n"
            output_log += f"Expected: {expected}\n"
            output_log += f"Returned: {result}\n"
            output_log += "✅ PASS\n" if passed else "❌ FAIL\n"

        except Exception as e:
            if isinstance(expected, str) and expected in type(e).__name__:
                output_log += f"\nTest Case {i}: {desc}\n"
                output_log += f"Input: {inputs}\n"
                output_log += f"Expected Exception: {expected}\n"
                output_log += f"Got Exception: {type(e).__name__}\n"
                output_log += "✅ PASS\n"
            else:
                output_log += f"\nTest Case {i}: {desc}\n"
                output_log += f"Input: {inputs}\n"
                output_log += "❌ ERROR\n"
                output_log += traceback.format_exc()

    return output_log

In [ ]:
def process_code(code, model_name, function_name):
    try:
        # Step 1 → Docstrings
        documented_code = add_docs_and_comments(code, model_name)

        # 👇 Update UI immediately
        yield documented_code, "", "⏳ Generating test cases..."

        # Step 2 → Test cases (use documented code)
        test_cases = generate_test_cases(documented_code, model_name)
        test_cases_str = json.dumps(test_cases, indent=2)

        # 👇 Update UI again
        yield documented_code, test_cases_str, "⏳ Running tests..."

        # Step 3 → Execution
        try:
            results = run_tests(documented_code, test_cases, function_name)
        except:
            results = run_tests(code, test_cases, function_name)

        # 👇 Final update
        yield documented_code, test_cases_str, results

    except Exception as e:
        yield "ERROR", "ERROR", str(e)

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# 🧪 LLM Code Tester")

    with gr.Row():
        # LEFT SIDE → INPUTS
        with gr.Column(scale=1):
            code_input = gr.Textbox(
                label="Python Code",
                lines=20,
                placeholder="Paste your Python function here..."
            )

            function_name = gr.Textbox(
                label="Function Name",
                placeholder="e.g. add"
            )

            model_dropdown = gr.Dropdown(
                choices=models,
                label="Select Model",
                value=models[0]
            )

            run_button = gr.Button("Run Pipeline")

        # RIGHT SIDE → OUTPUTS
        with gr.Column(scale=1):
            doc_output = gr.Code(label="📘 Documented Code")

            test_output = gr.Code(label="🧾 Test Cases (JSON)")

            result_output = gr.Textbox(
                label="📊 Test Results",
                lines=20
            )

    run_button.click(
        fn=process_code,
        inputs=[code_input, model_dropdown, function_name],
        outputs=[doc_output, test_output, result_output]
    )

# ------------------ LAUNCH ------------------ #
demo.launch()